To extract the unique categories from the OSM and Foursquare datasets, you can use the following SQL commands. These commands create new tables containing distinct category information and export them to CSV files.

```sql
CREATE TABLE osm_categories AS
SELECT DISTINCT osm_class, osm_type FROM osm ;
\copy osm_categories TO 'osm_categories.csv' WITH ( FORMAT csv, HEADER, DELIMITER ',', QUOTE '"', ESCAPE '"', NULL '' );
```


Foursquare can be downloaded from the Foursquare website: https://docs.foursquare.com/data-products/docs/categories
(data/fsq_personalization-apis-movement-sdk-categories.csv)

Then we feed data/fsq_personalization-apis-movement-sdk-categories.csv, osm_categories.csv and https://wiki.openstreetmap.org/wiki/Map_features to `GPT 5.5 with High Intelligence` to generate a mapping between Foursquare categories and OSM categories. Resulted in a mapping file fsq_osm_category_mapping.csv. 

prompt: "I want you to add two columns to fsq_personalization-apis-movement-sdk-categories.csv that are osm_class, osm_type that they have the same meaning. If ou cannot find a match from osm please put nf_fsq for both osm_class, osm_type."

We provided 100k of the matching algorithm as evidence multiple times to refine and correct the category mapping. 

In [19]:
import pandas
fsq_osm_category_mapping = pandas.read_csv('../../data/info/fsq_osm_category_mapping_restricted.csv')
fsq_personalization = pandas.read_csv('../../data/info/fsq_personalization-apis-movement-sdk-categories.csv')

In [20]:
category_mapping_dict = {}
for index, row in fsq_osm_category_mapping.iterrows():
    category_mapping_dict[row['Category ID']] = {
                                            'Category Name': row['Category Name'],
                                            'Category Label': row['Category Label'],
                                            "osm_class": row['osm_class'],
                                             "osm_type": row['osm_type']}


In [21]:

# check that the two files have the same [Category ID, Category Name, Category Label] values 
f1 = fsq_osm_category_mapping[['Category ID', 'Category Name', 'Category Label']].sort_values(by=['Category ID', 'Category Name', 'Category Label']).reset_index(drop=True)
f2 = fsq_personalization[['Category ID', 'Category Name', 'Category Label']].sort_values(by=['Category ID', 'Category Name', 'Category Label']).reset_index(drop=True)
assert f1.equals(f2), "The two files do not have the same [Category ID, Category Name, Category Label] values. Please check the files for discrepancies."

In [22]:

import ast

def to_set(x):
    if x is None:
        return set()

    if isinstance(x, set):
        return x

    if isinstance(x, (list, tuple)):
        return set(x)

    if isinstance(x, str):
        x = x.strip()

        if not x:
            return set()

        # Handles strings like "{'leisure', 'tourism'}" or "['leisure']"
        try:
            parsed = ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return {x}   # plain string like "leisure"

        if isinstance(parsed, str):
            return {parsed}

        return set(parsed)

    return {x}

def to_list(x):
    if x is None:
        return []

    if isinstance(x, list):
        return x

    if isinstance(x, (set, tuple)):
        return list(x)

    if isinstance(x, str):
        x = x.strip()

        if not x:
            return []

        # Handles strings like "{'leisure', 'tourism'}" or "['leisure']"
        try:
            parsed = ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return [x]   # plain string like "leisure"

        if isinstance(parsed, str):
            return [parsed]

        return list(parsed)

    return [x]

fast_category_mapping = {
    str(cat_id): {
        "osm_class": to_list(v.get("osm_class", [])),
        "osm_type": to_list(v.get("osm_type", []))
    }
    for cat_id, v in category_mapping_dict.items()
}


def parse_cat_ids(value):
    if pandas.isna(value):
        return []

    if isinstance(value, list):
        return [str(x).strip().strip("'\"") for x in value]

    value = str(value).strip()

    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [str(x).strip().strip("'\"") for x in parsed]
    except Exception:
        pass

    return [
        x.strip().strip("'\"")
        for x in value.strip("[]").replace(",", " ").split()
    ]

In [23]:
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath(".."))
import utils.dtype as dtype_utils

world_poi_path = "../../data/World_POI_levenshtein_1.0_0.00025.csv"

world_poi_df = pd.read_csv(
    world_poi_path,
    chunksize=1000,
    dtype=dtype_utils.get_dtype(which="lev")
)

def get_osm_class_matched(fsq_category_ids, osm_class):
    cat_ids = parse_cat_ids(fsq_category_ids)

    for cat_id in cat_ids:
        mapping = fast_category_mapping.get(cat_id)
        if mapping is None:
            continue

        if osm_class in mapping.get("osm_class", []):
            return "yes"

    return "no"


def get_osm_type_matched(fsq_category_ids, osm_type):
    cat_ids = parse_cat_ids(fsq_category_ids)

    for cat_id in cat_ids:
        mapping = fast_category_mapping.get(cat_id)
        if mapping is None:
            continue

        if osm_type in mapping.get("osm_type", []):
            return "yes"

    return "no"


output_path = world_poi_path.replace(".csv", "_category_matched.csv")
first_chunk = True

for chunk in world_poi_df:
    chunk = chunk.dropna(subset=["fsq_category_ids"]).copy()

    chunk["osm_class_matched"] = [
        get_osm_class_matched(fsq_ids, osm_class)
        for fsq_ids, osm_class in zip(
            chunk["fsq_category_ids"],
            chunk["osm_class"]
        )
    ]

    chunk["osm_type_matched"] = [
        get_osm_type_matched(fsq_ids, osm_type)
        for fsq_ids, osm_type in zip(
            chunk["fsq_category_ids"],
            chunk["osm_type"]
        )
    ]

    chunk.to_csv(
        output_path,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False
    break